In [1]:
import os
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.firefox.service import Service as FirefoxService
from selenium.webdriver.edge.service import Service as EdgeService
import re

def normalize(text):
    return re.sub(r"[^\w\s]", "", text.lower().strip())

try:
    df = pd.read_excel("../22127444_AutomationTesting_10/22127444_Data.xlsx", sheet_name="Contact")
    print("Excel loaded successfully. Total test cases:", df.shape[0])
except Exception as e:
    print("Error loading Excel:", e)
    exit(1)

edge_driver_path = r".\msedgedriver.exe"
browsers = {
    "edge": lambda: webdriver.Edge(service=EdgeService(executable_path=edge_driver_path)),
    "firefox": lambda: webdriver.Firefox(service=FirefoxService()),
    "chrome": lambda: webdriver.Chrome(service=ChromeService()),
}

os.makedirs("logs", exist_ok=True)
ATTACHMENT_DIR = os.getcwd()

for browser_name, browser_func in browsers.items():
    print(f"\nRunning tests on: {browser_name.upper()}")
    log_file = open(f"logs/contact_results_{browser_name}.txt", "w", encoding="utf-8")

    def log(msg):
        print(msg)
        log_file.write(msg + "\n")

    for idx, row in df.iterrows():
        row = {k: ("" if pd.isna(v) else str(v).strip()) for k, v in row.items()}
        test_id = row.get("Test case ID", f"TC_{idx+1}")
        log(f"\n[{test_id}] Starting test...")

        try:
            driver = browser_func()
            is_logged_in = str(row.get("Is Login", "")).strip().upper() == "TRUE"

            if is_logged_in:
                driver.get("http://localhost:4200/#/auth/login")
                WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, "email")))

                driver.find_element(By.ID, "email").send_keys("customer@practicesoftwaretesting.com")
                driver.find_element(By.CSS_SELECTOR, "[data-test='password']").send_keys("welcome01")
                driver.find_element(By.CSS_SELECTOR, "[data-test='login-submit']").click()

                WebDriverWait(driver, 10).until(
                    EC.url_contains("/account")
                )


            driver.get("http://localhost:4200/#/contact")
            WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, "message")))

            driver.execute_script("window.scrollTo(0, 0)")
            time.sleep(0.5)
            
            driver.execute_script("window.scrollTo(0, 0)")
            time.sleep(0.5)

            if not is_logged_in:
                WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, "first_name")))
                driver.find_element(By.ID, "first_name").send_keys(str(row.get("First_name", "")))
                driver.find_element(By.ID, "last_name").send_keys(str(row.get("Last_name", "")))
                driver.find_element(By.ID, "email").send_keys(str(row.get("Email", "")))


            subject_val = str(row.get("Subject", "")).strip()
            if subject_val.lower() not in ["", "nan"]:
                subject_elem = WebDriverWait(driver, 10).until(
                    EC.presence_of_element_located((By.ID, "subject"))
                )
                select = Select(subject_elem)
                found_option = False

                for option in select.options:
                    if option.text.strip().lower() == subject_val.lower():
                        option.click()
                        found_option = True
                        break

                if not found_option:
                    log(f"[{test_id}] Subject value not found in dropdown: '{subject_val}'")
                    log(f"[{test_id}] Available options:")
                    for option in select.options:
                        log(f"  - '{option.text.strip()}'")


            message_val = str(row.get("Message", "")).strip()
            if message_val:
                message_elem = driver.find_element(By.ID, "message")
                message_elem.clear()  
                message_elem.send_keys(message_val)

            attachment_name = str(row.get("Attachment", "")).strip()
            if attachment_name:
                file_path = os.path.join(ATTACHMENT_DIR, attachment_name)
                if os.path.exists(file_path):
                    driver.find_element(By.ID, "attachment").send_keys(file_path)
                else:
                    log(f"[{test_id}] Attachment not found: {file_path}")

            submit_button = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, "[data-test='contact-submit']"))
            )
            driver.execute_script("arguments[0].scrollIntoView(true);", submit_button)
            time.sleep(0.5)
            submit_button.click()
            time.sleep(1) 

            body_text = driver.find_element(By.TAG_NAME, "body").text.lower()
            expected_raw = row.get("expected_result", "")
            expected_lines = [
                line.strip().lower().replace('""', '"')
                for line in re.split(r"[\n\r]+", str(expected_raw))
                if line.strip()
            ]

            # case thành công
            if "thanks for your message" in body_text:
                if any("thanks for your message" in line for line in expected_lines):
                    log(f"[{test_id}] PASS | Success message shown.")
                else:
                    log(f"[{test_id}] FAIL | Unexpected success message.")
                    screenshot_path = f"logs/{test_id}_{browser_name}_fail.png"
                    driver.save_screenshot(screenshot_path)
                    log(f"[{test_id}] Screenshot saved to {screenshot_path}")
            else:
                # vẫn ở trang cũ
                all_found = True
                for expected_line in expected_lines:
                    if normalize(expected_line) not in normalize(body_text):
                        log(f"[{test_id}] FAIL | Missing expected: '{expected_line}'")
                        all_found = False

                if all_found:
                    log(f"[{test_id}] PASS")
                else:
                    screenshot_path = f"logs/{test_id}_{browser_name}_fail.png"
                    driver.save_screenshot(screenshot_path)
                    log(f"[{test_id}] Screenshot saved to {screenshot_path}")


        except Exception as e:
            log(f"[{test_id}] EXCEPTION: {e}")
            screenshot_path = f"logs/{test_id}_{browser_name}_exception.png"
            driver.save_screenshot(screenshot_path)
            log(f"[{test_id}] Screenshot saved to {screenshot_path}")

        finally:
            driver.quit()

    log_file.close()


Excel loaded successfully. Total test cases: 30

Running tests on: EDGE

[TC01] Starting test...
[TC01] FAIL | Missing expected: 'thanks for your message! we will contact you shortly.'
[TC01] Screenshot saved to logs/TC01_edge_fail.png

[TC02] Starting test...
[TC02] PASS

[TC03] Starting test...
[TC03] PASS

[TC04] Starting test...
[TC04] PASS

[TC05] Starting test...
[TC05] FAIL | Missing expected: 'the name field must not be greater than 120 characters'
[TC05] Screenshot saved to logs/TC05_edge_fail.png

[TC06] Starting test...
[TC06] FAIL | Missing expected: 'invalid email'
[TC06] Screenshot saved to logs/TC06_edge_fail.png

[TC07] Starting test...
[TC07] PASS

[TC08] Starting test...
[TC08] FAIL | Missing expected: 'the email field must not be greater than 120 characters'
[TC08] Screenshot saved to logs/TC08_edge_fail.png

[TC09] Starting test...
[TC09] PASS

[TC10] Starting test...
[TC10] Subject value not found in dropdown: 'Error 2: Translation error'
[TC10] Available options:


The geckodriver version (0.34.0) detected in PATH at geckodriver.EXE might not be compatible with the detected firefox version (141.0); currently, geckodriver 0.36.0 is recommended for firefox 141.*, so it is advised to delete the driver in PATH and retry


[TC01] FAIL | Missing expected: 'thanks for your message! we will contact you shortly.'
[TC01] Screenshot saved to logs/TC01_firefox_fail.png

[TC02] Starting test...


The geckodriver version (0.34.0) detected in PATH at geckodriver.EXE might not be compatible with the detected firefox version (141.0); currently, geckodriver 0.36.0 is recommended for firefox 141.*, so it is advised to delete the driver in PATH and retry


[TC02] PASS


...


[TC28] Starting test...
[TC28] PASS


The geckodriver version (0.34.0) detected in PATH at geckodriver.EXE might not be compatible with the detected firefox version (141.0); currently, geckodriver 0.36.0 is recommended for firefox 141.*, so it is advised to delete the driver in PATH and retry



[TC29] Starting test...
[TC29] FAIL | Missing expected: 'thanks for your message! we will contact you shortly.'
[TC29] Screenshot saved to logs/TC29_firefox_fail.png


The geckodriver version (0.34.0) detected in PATH at geckodriver.EXE might not be compatible with the detected firefox version (141.0); currently, geckodriver 0.36.0 is recommended for firefox 141.*, so it is advised to delete the driver in PATH and retry



[TC30] Starting test...
[TC30] FAIL | Missing expected: 'thanks for your message! we will contact you shortly.'
[TC30] Screenshot saved to logs/TC30_firefox_fail.png

Running tests on: CHROME

[TC01] Starting test...
[TC01] FAIL | Missing expected: 'thanks for your message! we will contact you shortly.'
[TC01] Screenshot saved to logs/TC01_chrome_fail.png

[TC02] Starting test...
[TC02] PASS

[TC03] Starting test...
[TC03] PASS

[TC04] Starting test...
[TC04] PASS

[TC05] Starting test...
[TC05] FAIL | Missing expected: 'the name field must not be greater than 120 characters'
[TC05] Screenshot saved to logs/TC05_chrome_fail.png

[TC06] Starting test...
[TC06] FAIL | Missing expected: 'invalid email'
[TC06] Screenshot saved to logs/TC06_chrome_fail.png

[TC07] Starting test...
[TC07] PASS

[TC08] Starting test...
[TC08] FAIL | Missing expected: 'the email field must not be greater than 120 characters'
[TC08] Screenshot saved to logs/TC08_chrome_fail.png

[TC09] Starting test...
[TC09] 

In [4]:
import os
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.firefox.service import Service as FirefoxService
from selenium.webdriver.edge.service import Service as EdgeService
import re

def normalize(text):
    return re.sub(r"[^\w\s]", "", text.lower().strip())

try:
    df_category = pd.read_excel("../22127444_AutomationTesting_10/22127444_Data.xlsx", sheet_name="AddCategory")
    print("Excel loaded successfully. Total category test cases:", df_category.shape[0])
except Exception as e:
    print("Error loading Excel:", e)
    exit(1)

edge_driver_path = r".\msedgedriver.exe"
browsers = {
    "edge": lambda: webdriver.Edge(service=EdgeService(executable_path=edge_driver_path)),
    "firefox": lambda: webdriver.Firefox(service=FirefoxService()),
    "chrome": lambda: webdriver.Chrome(service=ChromeService()),
}

os.makedirs("logs", exist_ok=True)
ATTACHMENT_DIR = os.getcwd()
for browser_name, browser_func in browsers.items():
    print(f"\nRunning category tests on: {browser_name.upper()}")
    log_file = open(f"logs/category_results_{browser_name}.txt", "w", encoding="utf-8")

    def log(msg):
        print(msg)
        log_file.write(msg + "\n")

    for idx, row in df_category.iterrows():
        row = {k: ("" if pd.isna(v) else str(v).strip()) for k, v in row.items()}
        test_id = row.get("Test case ID", f"TC_{idx+1}")
        log(f"\n[{test_id}] Starting test...")

        try:
            driver = browser_func()

            driver.get("http://localhost:4200/#/auth/login")
            WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, "email")))
            driver.find_element(By.ID, "email").send_keys("admin@practicesoftwaretesting.com")
            driver.find_element(By.CSS_SELECTOR, "[data-test='password']").send_keys("welcome01")
            driver.find_element(By.CSS_SELECTOR, "[data-test='login-submit']").click()
            WebDriverWait(driver, 10).until(EC.url_contains("/admin"))

            driver.get("http://localhost:4200/#/admin/categories/add")
            WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, "name")))

            name_val = row.get("name", "")
            slug_val = row.get("slug", "")
            parent_id_val = row.get("parent_id", "")

            driver.find_element(By.ID, "name").send_keys(name_val)
            driver.find_element(By.ID, "slug").send_keys(slug_val)

            if parent_id_val:
                select_elem = Select(driver.find_element(By.ID, "parent_id"))
                found = False
                for option in select_elem.options:
                    if parent_id_val in option.get_attribute("value"):
                        select_elem.select_by_value(option.get_attribute("value"))
                        found = True
                        break
                if not found:
                    log(f"[{test_id}] Parent ID '{parent_id_val}' not found in dropdown")

            # Submit form
            submit_button = driver.find_element(By.CSS_SELECTOR, "[data-test='category-submit']")
            submit_button.click()
            time.sleep(1)

            body_text = driver.find_element(By.TAG_NAME, "body").text.lower()
            expected_raw = row.get("expected_result", "")
            expected_lines = [normalize(line) for line in expected_raw.lower().splitlines() if line.strip()]

            # Evaluate result
            if any("category added" in line for line in expected_lines):
                if "category added" in body_text:
                    log(f"[{test_id}] PASS | Category added.")
                else:
                    log(f"[{test_id}] FAIL | Expected category to be added.")
                    screenshot_path = f"logs/{test_id}_{browser_name}_fail.png"
                    driver.save_screenshot(screenshot_path)
                    log(f"[{test_id}] Screenshot saved to {screenshot_path}")
            else:
                all_found = True
                for expected_line in expected_lines:
                    if normalize(expected_line) not in normalize(body_text):
                        log(f"[{test_id}] FAIL | Missing expected: '{expected_line}'")
                        all_found = False

                if all_found:
                    log(f"[{test_id}] PASS")
                else:
                    screenshot_path = f"logs/{test_id}_{browser_name}_fail.png"
                    driver.save_screenshot(screenshot_path)
                    log(f"[{test_id}] Screenshot saved to {screenshot_path}")

        except Exception as e:
            log(f"[{test_id}] EXCEPTION: {e}")
            screenshot_path = f"logs-2/{test_id}_{browser_name}_exception.png"
            driver.save_screenshot(screenshot_path)
            log(f"[{test_id}] Screenshot saved to {screenshot_path}")

        finally:
            driver.quit()

    log_file.close()


Excel loaded successfully. Total category test cases: 23

Running category tests on: EDGE

[TC01] Starting test...
[TC01] FAIL | Missing expected: 'category saved'
[TC01] Screenshot saved to logs/TC01_edge_fail.png

[TC02] Starting test...
[TC02] FAIL | Missing expected: 'category name already exists'
[TC02] Screenshot saved to logs/TC02_edge_fail.png

[TC03] Starting test...
[TC03] PASS

[TC04] Starting test...
[TC04] FAIL | Missing expected: 'category saved'
[TC04] Screenshot saved to logs/TC04_edge_fail.png

[TC05] Starting test...
[TC05] FAIL | Missing expected: 'name must be at most 128 characters'
[TC05] Screenshot saved to logs/TC05_edge_fail.png

[TC06] Starting test...
[TC06] PASS

[TC07] Starting test...
[TC07] FAIL | Missing expected: 'slug must be urlsafe'
[TC07] Screenshot saved to logs/TC07_edge_fail.png

[TC08] Starting test...
[TC08] FAIL | Missing expected: 'slug must be urlsafe'
[TC08] Screenshot saved to logs/TC08_edge_fail.png

[TC09] Starting test...
[TC09] FAIL | 

The geckodriver version (0.34.0) detected in PATH at geckodriver.EXE might not be compatible with the detected firefox version (141.0); currently, geckodriver 0.36.0 is recommended for firefox 141.*, so it is advised to delete the driver in PATH and retry


[TC01] FAIL | Missing expected: 'category saved'
[TC01] Screenshot saved to logs/TC01_firefox_fail.png


The geckodriver version (0.34.0) detected in PATH at geckodriver.EXE might not be compatible with the detected firefox version (141.0); currently, geckodriver 0.36.0 is recommended for firefox 141.*, so it is advised to delete the driver in PATH and retry



[TC02] Starting test...
[TC02] FAIL | Missing expected: 'category name already exists'
[TC02] Screenshot saved to logs/TC02_firefox_fail.png


...


[TC21] Starting test...
[TC21] FAIL | Missing expected: 'category saved'
[TC21] Screenshot saved to logs/TC21_firefox_fail.png


The geckodriver version (0.34.0) detected in PATH at geckodriver.EXE might not be compatible with the detected firefox version (141.0); currently, geckodriver 0.36.0 is recommended for firefox 141.*, so it is advised to delete the driver in PATH and retry



[TC22] Starting test...
[TC22] FAIL | Missing expected: 'category saved'
[TC22] Screenshot saved to logs/TC22_firefox_fail.png


The geckodriver version (0.34.0) detected in PATH at geckodriver.EXE might not be compatible with the detected firefox version (141.0); currently, geckodriver 0.36.0 is recommended for firefox 141.*, so it is advised to delete the driver in PATH and retry



[TC23] Starting test...
[TC23] FAIL | Missing expected: 'category saved'
[TC23] Screenshot saved to logs/TC23_firefox_fail.png

Running category tests on: CHROME

[TC01] Starting test...
[TC01] FAIL | Missing expected: 'category saved'
[TC01] Screenshot saved to logs/TC01_chrome_fail.png

[TC02] Starting test...
[TC02] FAIL | Missing expected: 'category name already exists'
[TC02] Screenshot saved to logs/TC02_chrome_fail.png

[TC03] Starting test...
[TC03] PASS

[TC04] Starting test...
[TC04] FAIL | Missing expected: 'category saved'
[TC04] Screenshot saved to logs/TC04_chrome_fail.png

[TC05] Starting test...
[TC05] FAIL | Missing expected: 'name must be at most 128 characters'
[TC05] Screenshot saved to logs/TC05_chrome_fail.png

[TC06] Starting test...
[TC06] PASS

[TC07] Starting test...
[TC07] FAIL | Missing expected: 'slug must be urlsafe'
[TC07] Screenshot saved to logs/TC07_chrome_fail.png

[TC08] Starting test...
[TC08] FAIL | Missing expected: 'slug must be urlsafe'
[TC08] 